In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

import gc,json

In [ ]:
# We run IBM services
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="", 
    overwrite=True,
    set_as_default=True,
)

service = QiskitRuntimeService()
backend = service.backend('ibm_fez')
pm = generate_preset_pass_manager(backend=backend, optimization_level=0)

In [ ]:
# Indicate qubit mapping
coupling_map = backend.coupling_map.get_edges()

disjoint_pairs = []
used_qubits = set()

for edge in coupling_map:
    q0, q1 = edge
    if q0 not in used_qubits and q1 not in used_qubits:
        disjoint_pairs.append((q0, q1))
        used_qubits.add(q0)
        used_qubits.add(q1)

print(f"Found {len(disjoint_pairs)} independent pairs on the processor.")

# Experiment

Because the fidelity test is performed in the same way for both test and use rounds, we use the same quantum circuit in all rounds and distinguish between them only at the postprocessing stage.

In [ ]:
# We need to create small chunks of circuits to not overload
# the quantum processor
N = 10000 # Total number of circuits you want to run
chunk_size = 2000 
num_chunks = int(N/ chunk_size)
num_qubits = backend.num_qubits

In [ ]:
# Save the data
checkpoint_file = "data/ibm_job_checkpoints_fez.jsonl" 

sampler = Sampler(mode=backend)
sampler.options.default_shots = 1

for chunk_idx in range(num_chunks):
    print(f"\n--- Processing Chunk {chunk_idx + 1} / {num_chunks} ---")

    chunk_circuits = []
    chunk_bases = []
    for _ in range(chunk_size):
        qc = QuantumCircuit(num_qubits, num_qubits)
        current_round_bases = {}
        
        for pair_idx, (q0, q1) in enumerate(disjoint_pairs):
            qc.h(q0)
            qc.cx(q0, q1)
            
            basis_choice = np.random.choice(['XX', 'YY', 'ZZ'])
            current_round_bases[pair_idx] = basis_choice
            
            if basis_choice == 'XX':
                qc.h(q0)
                qc.h(q1)
            elif basis_choice == 'YY':
                qc.sdg(q0)
                qc.h(q0)
                qc.sdg(q1)
                qc.h(q1)
                
            qc.measure([q0, q1], [q0, q1])
                
        chunk_circuits.append(qc)
        chunk_bases.append(current_round_bases)
    
    print(f"Transpiling {chunk_size} circuits...")
    transpiled_chunk = pm.run(chunk_circuits)
        
    print("Submitting to QPU...")
    job = sampler.run(transpiled_chunk)
    job_id = job.job_id()
    
    # Save data after running
    checkpoint_data = {
        "chunk_idx": chunk_idx,
        "job_id": job_id,
        "bases": chunk_bases
    }
    with open(checkpoint_file, 'a') as f:
        f.write(json.dumps(checkpoint_data) + '\n')
        
    print(f"Queued successfully. ID: {job_id}")
        
    # free memory
    del chunk_circuits
    del transpiled_chunk
    gc.collect()

print("\n=== All chunks have been submitted ===")

You can either save your own data or you can use the data already computed for this work

In [ ]:
# import json

# #  print("Connecting to IBM Quantum...")
# service = QiskitRuntimeService(channel="ibm_quantum_platform", token="")

# protocol_data = []
# successful_jobs_count = 0

# checkpoint_file = f"data/ibm_job_checkpoints_fez.jsonl"

# with open(checkpoint_file, 'r') as f:
#     for line in f:
#         data = json.loads(line.strip())
#         job_id = data["job_id"]
#         chunk_bases = data["bases"]  # List of 5000 basis dictionaries
        
#         # Fetch job status
#         job = service.job(job_id)
#         status = str(job.status())
        
#         if status in ["DONE", "JobStatus.DONE"]:
#             print(f"[+] Job {job_id} is DONE. Downloading results...")
#             successful_jobs_count += 1
            
#             result = job.result()
                
#             for circuit_idx, pub_result in enumerate(result):
#                 creg_name = list(pub_result.data.keys())[0] 
#                 bitstring = pub_result.data[creg_name].get_bitstrings()[0]
                
#                 circuit_basis = chunk_bases[circuit_idx]
                
#                 protocol_data.append({
#                     "base": circuit_basis,
#                     "bitstring": bitstring
#                 })
#         else:
#             print(f"[-] Job {job_id} skipped. Status: {status}")
            

# # Save data
# output_file = "data/complete_quantum_data.json"

# with open(output_file, 'w') as f:
#     json.dump(protocol_data, f)

# print("\n=== Download Complete ===")
# print(f"Successfully downloaded {successful_jobs_count} jobs.")
# print(f"Saved {len(protocol_data)} total circuit results to {output_file}.")

with open("data/complete_quantum_data.json", "r") as f:
    protocol_data = json.load(f)

num_circuits = len(protocol_data)
print(f"Number of circuits in protocol_data: {num_circuits}")

# Postprocessing

Here we decide how many rounds will be dedicated to test or use rounds according to $p$

In [ ]:
# We define the bounds
L1_Phi_plus = 0.75 
p = 0.10 
delta = 0.05
A = min(-0.25, -(1-p)/p * L1_Phi_plus) 
B = max(0.75, (1-p)/p * L1_Phi_plus)
Delta_p = B - A
Delta_p

In [ ]:
F_est_list = []
F_exp_list = []
epsilon_list = []
epsilon2_list = []

for pair_idx, (q0, q1) in enumerate(disjoint_pairs):
    S_T_X = [] ; S_Use_Fidelity = []
    
    for entry in protocol_data:
        bitstring = entry["bitstring"]
        
        base = entry["base"][str(pair_idx)]
        
        bit_q0 = bitstring[-1 - q0]
        bit_q1 = bitstring[-1 - q1]
        
        num_ones = (bit_q0 == '1') + (bit_q1 == '1')
        yt = 1 if num_ones % 2 == 0 else -1
        
        is_test_round = np.random.binomial(1, p) == 1
        
        # We calculate the estimator for both
        sign_alpha = -1 if base == 'YY' else 1
        Xt = 1/4 + yt * L1_Phi_plus * sign_alpha 

        if is_test_round:
            S_T_X.append(Xt)
        else:
            S_Use_Fidelity.append(Xt)
            
    # Compute the final verification statistics for this specific pair
    num_use = len(S_Use_Fidelity)
    num_tested = len(S_T_X)
    N_total = num_use + num_tested
    
    avg_X = (1 - p) / p * (sum(S_T_X) / num_use) 
    avg_X_use =(sum(S_Use_Fidelity) / num_use) 
    empirical_deviation_bound = (N_total * Delta_p / num_use) * np.sqrt(np.log(2/delta) / (2 * N_total)) 
    bound_ver = L1_Phi_plus*np.sqrt(2/num_use *np.log(2/delta))

    F_est_list.append(avg_X)
    F_exp_list.append(avg_X_use)
    epsilon_list.append(empirical_deviation_bound)
    epsilon2_list.append(bound_ver)

In [ ]:
# save data to plot results in julia
# np.savez(f"data/fidelity_p{p}.npz", 
#          F_est=F_est_list, 
#          F_exp=F_exp_list, 
#          epsilon=epsilon_list,
#          epsilon_exact = epsilon2_list)